# **Demo 01: Building a CrewAI-Powered Agent With Dynamic Web Search Using Hugging Face / Groq**

**Objective:** To demonstrate how to build a multi-agent system using CrewAI that intelligently identifies queries requiring real-time information and retrieves the most up-to-date data using a Hugging Face or Groq-hosted LLM and the Tavily web search API.

**Prerequisites:** Hugging Face API key (`HF_API_KEY`) or Groq API key (`GROQ_API_KEY`), Tavily API key

**Tools required:** Python 3.10+, CrewAI, Tavily Search API

**Scenario:** A product manager at a digital knowledge platform needs to manage an increasing volume of user queries, many of which require real-time updates and dynamic information. At present, these queries are reviewed and answered manually, leading to delays, inconsistencies, and rising operational costs. The goal is to automate the query-handling process using a multi-agent system that can identify when real-time data retrieval is needed and provide users with fast, accurate, and context-aware responses while maintaining efficiency and cost control. 

In [ ]:
# Step 1: Install the required libraries
# crewai — multi-agent orchestration framework
# langchain / langchain-openai / langchain-community / langchain-tavily — LLM and tool integrations
# tavily-python — client for the Tavily web-search API (provides real-time search results)
# pydantic — data validation and schema definitions used by CrewAI tools
# litellm — unified LLM gateway that lets us swap providers (Hugging Face, Groq, etc.)
%pip install --upgrade crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic
%pip install --upgrade litellm

Defaulting to user installation because normal site-packages is not writeable
  Using cached pydantic-2.13.3-py3-none-any.whl.metadata (108 kB)
  Using cached openai-2.32.0-py3-none-any.whl.metadata (31 kB)
Using cached openai-2.32.0-py3-none-any.whl (1.2 MB)
  Attempting uninstall: openai
    Found existing installation: openai 2.24.0
    Uninstalling openai-2.24.0:
      Successfully uninstalled openai-2.24.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\AgrawalN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached litellm-1.83.13-py3-none-any.whl.metadata (33 kB)
  Using cached openai-2.24.0-py3-none-any.whl.metadata (29 kB)
  Using cached python_dotenv-1.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-win_amd64.whl.metadata (7.4 kB)
Using cached litellm-1.83.13-py3-none-any.whl (16.4 MB)
Using cached openai-2.24.0-py3-none-any.whl (1.1 MB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached python_dotenv-1.0.1-py3-none-any.whl (19 kB)
Using cached pydantic_core-2.41.5-cp312-cp312-win_amd64.whl (2.0 MB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.2
    Uninstalling python-dotenv-1.2.2:
      Successfully uninstalled python-dotenv-1.2.2
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.33.2
    Uninstalling 

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\AgrawalN\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python312\\site-packages\\litellm\\proxy\\_experimental\\out\\experimental\\api-playground\\__next.!KGRhc2hib2FyZCk.experimental.api-playground.__PAGE__.txt'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\AgrawalN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# Step 1b: Install the optional Azure AI Inference extras for CrewAI
%pip install "crewai[azure-ai-inference]"

Defaulting to user installation because normal site-packages is not writeable
  Using cached pydantic-2.11.10-py3-none-any.whl.metadata (68 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
Using cached pydantic-2.11.10-py3-none-any.whl (444 kB)
Using cached pydantic_core-2.33.2-cp312-cp312-win_amd64.whl (2.0 MB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.0.1
    Uninstalling python-dotenv-1.0.1:
      Successfully uninstalled python-dotenv-1.0.1
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.5
    Uninstalling pydantic-2.12.5:
      Successfully uninstalled pydantic-2.12.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 1.2.1 requires openai<3.0.0,>=2.26.0, but you have openai 2.24.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\AgrawalN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# Step 2: Import dependencies
# os          — read environment variables (API keys)
# requests    — make HTTP calls to the Tavily search API
# LLM         — CrewAI's LLM wrapper (supports multiple providers)
# Agent, Task, Crew — core CrewAI primitives for defining agents, tasks, and the crew
# BaseTool    — base class for building custom CrewAI tools
import os
import requests
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool

In [ ]:
# Step 3: Configure the Tavily search tool and the LLM provider

# ---- Tavily API Key ----
# Used by our custom search tool to call the Tavily Search API
TAVILY_API_KEY = "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"

# ---- Custom CrewAI Tool for Web Search ----
# Every CrewAI tool must inherit from BaseTool and implement _run().
# When an agent invokes this tool, it sends a POST request to the Tavily API
# and returns the top search results as formatted text.
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 5
        }

        response = requests.post(url, json=payload)
        data = response.json()

        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)


search_tool = TavilySearchTool()

# ---- LLM Provider Selection ----
# Set LLM_PROVIDER to "huggingface" or "groq" to switch between providers.
# Both use an OpenAI-compatible endpoint so CrewAI can interact with them uniformly.
LLM_PROVIDER = "groq"

if LLM_PROVIDER == "huggingface":
    # Hugging Face Inference API — free-tier model via the OpenAI-compatible router
    llm = LLM(
        model="Qwen/Qwen2.5-7B-Instruct",
        provider="openai",
        api_key=os.getenv("HF_API_KEY"),
        base_url="https://router.huggingface.co/v1",
        temperature=0.5,
    )
elif LLM_PROVIDER == "groq":
    # Groq — fast inference via the OpenAI-compatible endpoint
    llm = LLM(
        model="llama-3.3-70b-versatile",
        provider="openai",
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1",
        temperature=0.5,
    )
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}. Use 'huggingface' or 'groq'.")

In [ ]:
# Step 4: Define two agents — Researcher and Writer
# Each agent has a role, goal, backstory, and optional tools.
# verbose=True enables detailed logging of the agent's reasoning steps.
# allow_delegation=False means each agent works independently (no hierarchical delegation).

# Researcher agent — uses the Tavily search tool to find real-time information
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for FMCG",
    backstory="You are an expert in artificial intelligence and stay updated with the latest research trends in FMCG.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)

# Writer agent — synthesises the Researcher's findings into an executive summary
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory="You are an experienced technical writer with expertise in summarizing research for executives.",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [ ]:
# Step 5: Create tasks with dependencies
# Tasks are the prompts given to agents. Each task includes a description,
# an expected output format, and the agent responsible for executing it.

# Research task — the Researcher agent searches the web for the latest AI advancements in FMCG
task_research = Task(
    description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
    expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
    agent=researcher
)

# Writing task — the Writer agent consumes the Researcher's output (via context)
# and produces a concise executive summary
task_write = Task(
    description="""
Write a concise executive summary using the research notes.

Requirements:
- Maximum 200 words
- Use bullet points
- Focus only on the 3 key advancements
""",
    expected_output="Executive summary of AI advancements in FMCG.",
    agent=writer,
    context=[task_research]  # uses the output of the research task as context
)

In [ ]:
# Step 6: Assemble the Crew and execute
# The Crew ties agents and tasks together. Tasks run sequentially (default process).
# kickoff() starts the pipeline: Researcher runs first, then Writer uses its output.
crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

result = crew.kickoff()

print("\nFinal Output:\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ae712cf3-bed8-4c9b-a3e2-c0c0d821e026                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the web and identify the top 3 recent advancements in AI for FMCG.                                │
│  ID: 96843f70-f48b-49ea-accc-3aee1117dc06                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Search the web and identify the top 3 recent advancements in AI for FMCG.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'latest AI advancements in FMCG'}                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: AI Solutions for FMCG: Transform Operations & Boost Profit | Techelix - https://techelix.co/industry/ai-for-fmcg/
AI in FMCG: Top Use Cases You Need To Know - SmartDev - https://smartdev.com/ai-use-ca...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Executive Summary: AI Advancements in FMCG                                                                     │
│                                                                                                                 │
│  The Fast-Moving Consumer Goods (FMCG) industry is undergoing a significant transformation with the             │
│  integration of Artificial Intelligence (AI). Key advancements include:                                         │
│  * **Predictive Analytics**: AI-powered predictive analytics to forecast demand, manage inventory, and          │
│  optimize supply chain operations.                                                                              │
│  * **Computer Vision**: Computer vision to improve quality control, detect defects, and enhance customer        │
│  experience.                                                                                                    │
│  * **Natural Language Processing (NLP)**: NLP to analyze customer feedback, sentiment, and preferences,         │
│  enabling data-driven decisions to improve product development and marketing strategies.                        │
│                                                                                                                 │
│  These AI advancements are revolutionizing the FMCG industry by enhancing operational efficiency, improving     │
│  customer experience, and driving business growth.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Write a concise executive summary using the research notes.                                                    │
│                                                                                                                 │
│  Requirements:                                                                                                  │
│  - Maximum 200 words                                                                                            │
│  - Use bullet points                                                                                            │
│  - Focus only on the 3 key advancements                                                                         │
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ae712cf3-bed8-4c9b-a3e2-c0c0d821e026                                                                       │
│  Final Output: Executive Summary: AI Advancements in FMCG                                                       │
│                                                                                                                 │
│  The Fast-Moving Consumer Goods (FMCG) industry is undergoing a significant transformation with the             │
│  integration of Artificial Intelligence (AI). Key advancements include:                                         │
│  * **Predictive Analytics**: AI-powered predictive analytics to forecast demand, manage inventory, and          │
│  optimize supply chain operations.                                                                              │
│  * **Computer Vision**: Computer vision to improve quality control, detect defects, and enhance customer        │
│  experience.                                                                                                    │
│  * **Natural Language Processing (NLP)**: NLP to analyze customer feedback, sentiment, and preferences,         │
│  enabling data-driven decisions to improve product development and marketing strategies.                        │
│                                                                                                                 │
│  These AI advancements are revolutionizing the FMCG industry by enhancing operational efficiency, improving     │
│  customer experience, and driving business growth.                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final Output:

Executive Summary: AI Advancements in FMCG

The Fast-Moving Consumer Goods (FMCG) industry is undergoing a significant transformation with the integration of Artificial Intelligence (AI). Key advancements include:
* **Predictive Analytics**: AI-powered predictive analytics to forecast demand, manage inventory, and optimize supply chain operations.
* **Computer Vision**: Computer vision to improve quality control, detect defects, and enhance customer experience.
* **Natural Language Processing (NLP)**: NLP to analyze customer feedback, sentiment, and preferences, enabling data-driven decisions to improve product development and marketing strategies.

These AI advancements are revolutionizing the FMCG industry by enhancing operational efficiency, improving customer experience, and driving business growth.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯